# 05 · Cenários de incidente (v2)
**CRISP-DM:** Preparação dos Dados (gabarito de avaliação)

Cada **cenário** simula um incidente: uma degradação começa em t₀ e o incidente é registrado em t = t₀ + atraso. A pergunta é respondida **no instante t**, olhando só para trás.

| Motivo | Abrangência | Baixa | Média | Alta |
|---|---|---|---|---|
| Erro de negócio (4xx) | generalizado | 5% | 15% | 40% das respostas 2xx viram 400 |
| Erro de negócio (4xx) | 1 consumidor | 30% | 60% | 90% das 2xx do consumidor viram 400 |
| Erro de sistema (5xx) | generalizado | 5% | 15% | 40% das 2xx viram 503 |
| Erro de sistema (5xx) | 1 consumidor | 30% | 60% | 90% das 2xx do consumidor viram 503 |
| Latência | generalizada | ×1,5 | ×2,5 | ×4 no tempo de execução |

- **Atraso** entre o início da degradação e o registro: 15, 30, 60 ou 120 min.
- **Instante do registro:** qualquer minuto de 01/09 04:00 até o fim dos dados de cada recurso (08/09 à noite; 03/09 07:11 para quatro recursos).
- O consumidor afetado é sorteado com probabilidade proporcional ao seu volume.
- **5 repetições** por recurso × motivo × intensidade × atraso = **3.600 incidentes**, mais **3.600 consultas de controle** (instantes sem falha injetada).
- Cada cenário é independente: a falha é aplicada numa cópia dos registros por minuto do recurso.

In [1]:
%run ./00_funcoes.ipynb

00_funcoes (v2) carregado · /home/wsl/gh/mestrado/claude9/v2


In [2]:
brutos = pd.read_parquet(RESULTADOS / "02_brutos_consolidados.parquet")
B = Brutos(brutos)
fins = pd.Series(B.fim)
cenarios = gerar_cenarios(fins)
cenarios.to_csv(RESULTADOS / "05_cenarios.csv", index=False)
print(cenarios["tipo"].value_counts().to_dict())
cenarios.assign(recurso=cenarios["recurso"].map(rot)).groupby(["recurso", "tipo"]).size().unstack()

{'incidente': 3600, 'controle': 3600}


tipo,controle,incidente
recurso,,
E01,300,300
E02,300,300
E03,300,300
E04,300,300
E05,300,300
E06,300,300
E07,300,300
E08,300,300
E09,300,300


## Exemplo: efeito de um cenário nas faixas antes do registro

In [3]:
ex = cenarios[(cenarios["tipo"] == "incidente") & (cenarios["motivo"] == "5xx_geral") & (cenarios["intensidade"] == "media")
              & (cenarios["atraso"] == 30)].iloc[0]
sem = ex.copy()
sem["tipo"] = "controle"
com_falha = pd.DataFrame(periodos_do_cenario(B, ex))
sem_falha = pd.DataFrame(periodos_do_cenario(B, sem))
cols = ["periodo", "lo", "hi", "vol", "tx_4xx", "tx_5xx", "lat_p95", "n_cli_5xx"]
print("recurso", rot(ex["recurso"]), "· registro", ex["t_registro"], "· início real 30 min antes")
display(sem_falha[cols].round(3).rename(columns=lambda c: c + " (sem)" if c not in ("periodo", "lo", "hi") else c)
        .join(com_falha[["tx_5xx", "n_cli_5xx"]].round(3).rename(columns=lambda c: c + " (com)")))

recurso E12 · registro 2026-09-04 13:28:00 · início real 30 min antes


,periodo,lo,hi,vol (sem),tx_4xx (sem),tx_5xx (sem),lat_p95 (sem),n_cli_5xx (sem),tx_5xx (com),n_cli_5xx (com)
0,L,0,15,89,0.000,0.000,193.0,0,0.169,10
1,L,0,30,348,0.009,0.000,474.0,0,0.167,28
2,L,0,60,593,0.005,0.000,474.0,0,0.098,28
3,banda,0,15,89,0.000,0.000,193.0,0,0.169,10
4,banda,15,30,259,0.012,0.000,474.0,0,0.166,22
5,banda,30,60,245,0.000,0.000,270.0,0,0.000,0
6,banda,60,120,751,0.017,0.001,503.0,1,0.001,1
7,banda,120,180,572,0.003,0.002,444.0,1,0.002,1
8,banda,180,240,629,0.006,0.008,440.0,2,0.008,2
